# Higgs Boson Decaying to Tau Leptons

<div>
<img src="https://raw.githubusercontent.com/illinois-ipaml/MachineLearningForPhysics/main/img/Project_HiggsTauTau-EventDisplay.jpg" width=800></img>
</div>

*Event display from the ATLAS experiment showing a candidate Higgs boson decaying to a pair of τ leptons. Image: ATLAS Experiment © CERN.*

## <span style="color:Orange">Learning Objectives</span>

By the end of this project you should be able to:

* explain why the LHC can produce the Higgs boson and how the ATLAS and CMS detectors observe its decay products;
* describe the Higgs boson's role in the Standard Model, its main decay modes, and why $H\to\tau\tau$ is an important channel;
* load and inspect a large simulated collider dataset efficiently;
* connect each feature in the dataset to a physical quantity measured or reconstructed in a detector; and
* use histograms to judge which features best separate signal ($H\to\tau\tau$) from background ($Z\to\tau\tau$), the starting point for the machine-learning classifiers in Project 02.

## <span style="color:Orange">Overview</span>

In July 2012 the ATLAS and CMS collaborations at CERN's Large Hadron Collider (LHC) announced the discovery of a new particle with a mass near 125 GeV, consistent with the Higgs boson predicted by the Standard Model (SM). Neubauer's research group at Illinois contributed to the data analysis in the ATLAS discovery paper. François Englert and Peter Higgs received the 2013 Nobel Prize in Physics for the theoretical prediction. [[3]](https://doi.org/10.1016/j.physletb.2012.08.020), [[4]](https://doi.org/10.1016/j.physletb.2012.08.021), [[5]](https://www.nobelprize.org/prizes/physics/2013/summary/)

See this [video](https://youtu.be/__btEwKQB2I) for more on the central role that the Higgs boson plays in the SM and on Illinois researchers' role in its discovery.

The discovery relied mainly on the Higgs boson's decays to other bosons ($\gamma\gamma$, $ZZ^*$, $WW^*$). The SM also predicts that the Higgs boson couples to **fermions**, with strength proportional to the fermion's mass. Its decay to a pair of τ leptons, $H\to\tau^+\tau^-$, has a sizeable branching ratio and a manageable background, which makes it the most sensitive direct probe of the Higgs coupling to leptons. In 2014, when the reference papers for this project were written [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html), [[2]](https://doi.org/10.1103/PhysRevLett.114.111801), $H\to\tau\tau$ had not yet been observed at the 5σ level. It was established later by the combined ATLAS and CMS Run 1 analysis [[8]](https://doi.org/10.1007/JHEP08%282016%29045) and then by CMS alone [[9]](https://doi.org/10.1016/j.physletb.2018.02.004).

The main challenge is that the process $Z\to\tau^+\tau^-$ produces exactly the same visible final state and is far more common. In this project you will explore the simulated dataset that Baldi, Sadowski and Whiteson used to show that deep neural networks can separate the two processes better than traditional approaches [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html), [[2]](https://doi.org/10.1103/PhysRevLett.114.111801).

## <span style="color:Orange">Data Source</span>

**Higgs to Tau Tau dataset** (UCI MLPhysics portal, D. Whiteson) [[10]](https://mlphysics.ics.uci.edu/data/htautau/)

* Directory: https://mlphysics.ics.uci.edu/data/htautau/ (also read `readme.txt` there)
* **Signal** ($gg\to H\to\tau^+\tau^-$): https://mlphysics.ics.uci.edu/data/htautau/htautau.txt.gz (3.8 GB)
* **Background** ($Z\to\tau^+\tau^-$): https://mlphysics.ics.uci.edu/data/htautau/ztautau.txt.gz (3.7 GB)

**What's in the files**

* Signal and background are in **separate files**, and there is **no label column**. You have to create the label yourself (1 = signal, 0 = background).
* Each row is one simulated event with 25 **whitespace-separated** columns and **no header**. The first 10 columns are low-level features and the remaining 15 are high-level features computed from them. Values are in physical units (GeV for momenta and masses, radians for angles), not normalized.
* Only the selected events are included: those where both τ leptons decayed leptonically ($\tau\to\ell\nu\bar\nu$, with $\ell = e,\mu$), giving two charged leptons, missing transverse momentum from the neutrinos, and possibly jets.

**Practical tips**

* Together the files are about 7.5 GB compressed, so **don't download them in full**. The cell below streams and decompresses only the first `N_EVENTS` rows of each file over the network, which takes seconds.
* Plain `pd.read_csv(<URL>, nrows=...)` downloads the whole file before applying `nrows`. Wrapping the stream in `gzip.GzipFile` as below avoids this.
* The first rows of each file are as random a sample as any other rows, so a few hundred thousand events per class are plenty for this project.

In [ ]:
import gzip
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://mlphysics.ics.uci.edu/data/htautau/"
N_EVENTS = 250_000  # per class; increase once your code works

def read_events(filename, nrows):
    """Stream the first `nrows` events of a gzipped file without downloading all of it."""
    with urllib.request.urlopen(BASE_URL + filename) as response, gzip.GzipFile(fileobj=response) as f:
        return pd.read_csv(f, sep=r"\s+", header=None, nrows=nrows)

sig = read_events("htautau.txt.gz", N_EVENTS)
bkg = read_events("ztautau.txt.gz", N_EVENTS)

# The files have no header; name the columns f00..f24 until you identify them (Question 06).
columns = [f"f{i:02d}" for i in range(sig.shape[1])]
sig.columns = bkg.columns = columns
sig["label"], bkg["label"] = 1, 0
df = pd.concat([sig, bkg], ignore_index=True)
df.head()

## <span style="color:Orange">Questions</span>

Answer each question in a markdown cell directly below it (add code cells where needed). For the background questions in Part A, aim for one to two well-organized paragraphs, and **cite your sources**. The references at the end of this notebook are a good starting point. Your notebook should run from top to bottom without errors.

#### Part A: Physics background

### <span style="color:LightGreen">Question 01</span>

What is the Large Hadron Collider (LHC)? What about the LHC makes it possible to produce heavy particles like the Higgs boson? Include the relevant quantitative parameters, such as the proton–proton center-of-mass energy $\sqrt{s}$ and the luminosity, and explain why each one matters for producing a rare, heavy particle.

### <span style="color:LightGreen">Question 02</span>

The Higgs boson was the last particle predicted by the Standard Model (SM) to be discovered. In what ways does it play a particularly important role in the SM? Consider, for example, how the $W$ and $Z$ bosons and the fermions acquire mass.

### <span style="color:LightGreen">Question 03</span>

The Higgs boson is unstable and decays almost immediately. It has several possible decays ("decay modes"). List the main decay modes of a 125 GeV Higgs boson and the SM-predicted branching ratio of each in a table, citing your source (for example, Refs. [[6]](https://pdg.lbl.gov/2025/reviews/rpp2025-rev-higgs-boson.pdf) or [[7]](https://doi.org/10.23731/CYRM-2017-002)). No calculation is needed. Then briefly comment:

* Why is $H\to b\bar b$ the most common decay, yet the discovery was made mainly in the $H\to\gamma\gamma$ and $H\to ZZ^*\to 4\ell$ channels?
* Why is $H\to\tau\tau$ a particularly attractive channel for measuring the Higgs coupling to fermions?

### <span style="color:LightGreen">Question 04</span>

Briefly describe the ATLAS and CMS experiments that record LHC proton–proton collision data to study the Higgs boson. What are the main detector subsystems, and which particles does each one measure?

A τ lepton decays before it reaches the detector. How does a τ lepton show up in the detector when it decays leptonically, and when it decays hadronically? Why does $H\to\tau\tau$ always produce missing transverse momentum?

<hr style="border:1px solid gray; margin: 2em 0;">

#### Part B: Exploring the $H\to\tau\tau$ dataset

The remaining questions use the dataset described in the Data Source section above.

### <span style="color:LightGreen">Question 05</span>

How many signal events and how many background events does the full dataset contain, and how many columns does each file have? Explain how you determined the counts without loading the full dataset into memory. For example, you could stream each file in chunks with `pd.read_csv(..., chunksize=...)`, or count lines with `gunzip -c file | wc -l` if you have downloaded it. This takes a few minutes per file, so run it once and record the result. Compare your counts with the number of simulated events quoted in Ref. [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html).

### <span style="color:LightGreen">Question 06</span>

Read Section 2 of Ref. [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html) (also summarized in the `readme.txt` file in the data directory) and answer the following:

1. Write the **signal** and **background** processes as chains of decays down to the particles that reach the detector. Explain why the two processes leave the same set of final-state objects, and which physical differences between the Higgs boson and the $Z$ boson (mass, spin) could still distinguish them.
2. The paper lists the low-level features as the momenta of the two charged leptons, the missing transverse momentum, and jet information. It lists the high-level features as axial missing momentum, scalar sum of momenta, relative missing momentum, $\Delta\phi(\ell^+,\ell^-)$, $\Delta\eta(\ell^+,\ell^-)$, $\Delta R(\ell^+,\ell^-)$, $m_{\ell\ell}$, the missing-mass $m_{\rm MMC}$ [[11]](https://doi.org/10.1016/j.nima.2011.07.009), sphericity, transverse sphericity, and the invariant mass of all visible objects. Briefly explain what each quantity measures and why it might help separate $H\to\tau\tau$ from $Z\to\tau\tau$.
3. The data files **do not document the column order**. Use the value ranges of the columns (e.g. $p_T \geq 20$ GeV, $|\eta| < 2.5$, $-\pi \leq \phi \leq \pi$) and your physics knowledge to identify as many of the 25 columns as you can. Where possible, **verify** an identification by recomputing the quantity from other columns. For example, compute $\Delta R(\ell_1,\ell_2)$ or the dilepton invariant mass $m_{\ell\ell}^2 \approx 2\,p_{T1}\,p_{T2}\,[\cosh(\Delta\eta) - \cos(\Delta\phi)]$ from the lepton columns and compare. Summarize your results in a table, and rename the columns of your DataFrame accordingly.

### <span style="color:LightGreen">Question 07</span>

Plot histograms of all 25 features with signal and background **overlaid on the same axes**. Normalize each histogram to unit area (e.g., `density=True` with the same bin edges for both classes), and arrange the plots in a grid (e.g., 5 × 5). Use sensible ranges or log scales where long tails hide the structure, and label the axes with the quantity and its units.

Note that some columns use special values when a quantity is undefined. For example, look for spikes at exactly 0 or −1 in jet-related variables or $m_{\rm MMC}$. How should such values be handled when you plot them or feed them to a classifier?

Which features show the most significant differences between signal and background? Which show almost none? Explain the differences physically, using your answer to Question 06. Compare your plots with Figs. 2 and 3 of Ref. [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html).

*Optional extension:* rank the features by a single number that measures separation, for example the area under the ROC curve obtained when each feature is used alone as a discriminant (`sklearn.metrics.roc_auc_score`). Do the high-level features separate the classes better than the low-level ones?

## <span style="color:Orange">References</span>

__[<span style="color:Red">1</span>]__ P. Sadowski, D. Whiteson, P. Baldi, "Searching for Higgs Boson Decay Modes with Deep Learning," *Advances in Neural Information Processing Systems 27 (NIPS 2014)*. [Abstract](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html), [PDF](https://papers.nips.cc/paper_files/paper/2014/file/44f58a78b6dd03bf9ccb526e7c7538d5-Paper.pdf)

__[<span style="color:Red">2</span>]__ P. Baldi, P. Sadowski, D. Whiteson, "Enhanced Higgs Boson to τ⁺τ⁻ Search with Deep Learning," *Phys. Rev. Lett.* **114**, 111801 (2015). [doi:10.1103/PhysRevLett.114.111801](https://doi.org/10.1103/PhysRevLett.114.111801), [arXiv:1410.3469](https://arxiv.org/abs/1410.3469)

__[<span style="color:Red">3</span>]__ ATLAS Collaboration, "Observation of a new particle in the search for the Standard Model Higgs boson with the ATLAS detector at the LHC," *Phys. Lett. B* **716**, 1 (2012). [doi:10.1016/j.physletb.2012.08.020](https://doi.org/10.1016/j.physletb.2012.08.020), [arXiv:1207.7214](https://arxiv.org/abs/1207.7214)

__[<span style="color:Red">4</span>]__ CMS Collaboration, "Observation of a new boson at a mass of 125 GeV with the CMS experiment at the LHC," *Phys. Lett. B* **716**, 30 (2012). [doi:10.1016/j.physletb.2012.08.021](https://doi.org/10.1016/j.physletb.2012.08.021), [arXiv:1207.7235](https://arxiv.org/abs/1207.7235)

__[<span style="color:Red">5</span>]__ The Nobel Prize in Physics 2013 (F. Englert and P. W. Higgs). https://www.nobelprize.org/prizes/physics/2013/summary/

__[<span style="color:Red">6</span>]__ Particle Data Group, "Status of Higgs Boson Physics," in *Review of Particle Physics* (2025). https://pdg.lbl.gov/2025/reviews/rpp2025-rev-higgs-boson.pdf

__[<span style="color:Red">7</span>]__ LHC Higgs Cross Section Working Group, "Handbook of LHC Higgs Cross Sections: 4. Deciphering the Nature of the Higgs Sector," CERN Yellow Report CERN-2017-002-M (2017). [doi:10.23731/CYRM-2017-002](https://doi.org/10.23731/CYRM-2017-002), [arXiv:1610.07922](https://arxiv.org/abs/1610.07922)

__[<span style="color:Red">8</span>]__ ATLAS and CMS Collaborations, "Measurements of the Higgs boson production and decay rates and constraints on its couplings from a combined ATLAS and CMS analysis of the LHC pp collision data at √s = 7 and 8 TeV," *JHEP* **08**, 045 (2016). [doi:10.1007/JHEP08(2016)045](https://doi.org/10.1007/JHEP08%282016%29045), [arXiv:1606.02266](https://arxiv.org/abs/1606.02266)

__[<span style="color:Red">9</span>]__ CMS Collaboration, "Observation of the Higgs boson decay to a pair of τ leptons with the CMS detector," *Phys. Lett. B* **779**, 283 (2018). [doi:10.1016/j.physletb.2018.02.004](https://doi.org/10.1016/j.physletb.2018.02.004), [arXiv:1708.00373](https://arxiv.org/abs/1708.00373)

__[<span style="color:Red">10</span>]__ D. Whiteson, Higgs to Tau Tau dataset, UCI MLPhysics data portal. https://mlphysics.ics.uci.edu/data/htautau/

__[<span style="color:Red">11</span>]__ A. Elagin, P. Murat, A. Pranko, A. Safonov, "A new mass reconstruction technique for resonances decaying to di-tau," *Nucl. Instrum. Meth. A* **654**, 481 (2011). [doi:10.1016/j.nima.2011.07.009](https://doi.org/10.1016/j.nima.2011.07.009), [arXiv:1012.4686](https://arxiv.org/abs/1012.4686)

__[<span style="color:Red">12</span>]__ ATLAS Collaboration, "Dataset from the ATLAS Higgs Boson Machine Learning Challenge 2014," CERN Open Data Portal. [doi:10.7483/OPENDATA.ATLAS.ZBP2.M5T8](https://doi.org/10.7483/OPENDATA.ATLAS.ZBP2.M5T8). This is a related $H\to\tau\tau$ dataset from full ATLAS detector simulation, for further exploration.

## <span style="color:Orange">Acknowledgements</span>

* Initial version: Mark Neubauer
* Revised: 2026

© Copyright 2026